In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('per_seed.csv')
df

,algoritma,seed,acc,val_acc,prec,val_prec,rec,val_rec,f1,val_f1,...,ronde_terbaik,n_client_eval,total_MB,waktu_s,f1_latih,auc_latih,prauc_latih,f1_non_latih,auc_non_latih,prauc_non_latih
0,fedavg,42,0.700014,0.701982,0.371047,0.386292,0.624613,0.612206,0.465542,0.473692,...,3,44,34.98528,112.5,0.465542,0.735731,0.429497,NaN,NaN,NaN
1,fedprox,42,0.697732,0.701785,0.369859,0.387598,0.632427,0.622953,0.466751,0.477868,...,3,44,34.98528,137.2,0.466751,0.737247,0.432111,NaN,NaN,NaN
2,fedpairwise_T0.01_d0.3,42,0.700320,0.706438,0.371941,0.392961,0.628357,0.624232,0.467284,0.482305,...,4,44,34.98528,116.9,0.467284,0.739680,0.436606,NaN,NaN,NaN
3,fedpairwise_T0.01_d0.5,42,0.701137,0.705261,0.373097,0.391322,0.630311,0.621929,0.468737,0.480383,...,4,44,34.98528,114.0,0.468737,0.739242,0.435466,NaN,NaN,NaN
4,fedpairwise_T0.01_d0,42,0.701137,0.706326,0.372285,0.392523,0.624939,0.621929,0.466606,0.481287,...,4,44,34.98528,97.4,0.466606,0.739417,0.437118,NaN,NaN,NaN
5,fedsim_g3,42,0.697392,0.701281,0.364641,0.382114,0.601660,0.589304,0.454082,0.463613,...,2,44,34.98528,83.7,0.454082,0.724534,0.411974,NaN,NaN,NaN
6,fedsim_g5,42,0.699196,0.701926,0.365973,0.382630,0.598079,0.587897,0.454085,0.463556,...,2,44,34.98528,82.6,0.454085,0.721058,0.409556,NaN,NaN,NaN
7,fedavg,0,0.703453,0.709745,0.374142,0.395041,0.620869,0.611566,0.466916,0.480016,...,3,44,34.98528,88.3,0.466916,0.736904,0.430946,NaN,NaN,NaN
8,fedprox,0,0.703010,0.709185,0.374011,0.394823,0.623148,0.614765,0.467456,0.480837,...,2,44,34.98528,133.1,0.467456,0.736542,0.432681,NaN,NaN,NaN
9,fedpairwise_T0.01_d0.3,0,0.705121,0.709157,0.377016,0.395375,0.628032,0.619115,0.471177,0.482573,...,3,44,34.98528,103.5,0.471177,0.739536,0.438635,NaN,NaN,NaN


In [2]:
# Metrik utama yang ingin dibandingkan
metrik = ["val_f1", "val_auc", "val_prauc", "val_prec", "val_rec"]

# 1) Tabel per seed: baris = algoritma, kolom = seed (untuk satu metrik)
pivot_f1 = df.pivot_table(index="algoritma", columns="seed", values="val_f1")
pivot_f1["mean"] = pivot_f1.mean(axis=1)
pivot_f1["std"] = pivot_f1.drop(columns="mean").std(axis=1)
print("=== val_f1 per seed ===")
print(pivot_f1.round(4))

# 2) Semua metrik sekaligus, per algoritma dan seed
tabel_semua = (
    df.set_index(["algoritma", "seed"])[metrik]
      .sort_index()
      .round(4)
)
print("\n=== Semua metrik per algoritma & seed ===")
print(tabel_semua)

# 3) Ringkasan rata-rata ± std lintas seed
ringkas = df.groupby("algoritma")[metrik].agg(["mean", "std"]).round(4)
print("\n=== Ringkasan lintas seed ===")
print(ringkas)

# 4) Ranking algoritma di setiap seed (1 = terbaik)
df["rank_val_f1"] = df.groupby("seed")["val_f1"].rank(ascending=False, method="min")
rank = df.pivot_table(index="algoritma", columns="seed", values="rank_val_f1")
rank["rata2_rank"] = rank.mean(axis=1)
print("\n=== Ranking val_f1 per seed (1 = terbaik) ===")
print(rank.sort_values("rata2_rank"))

# 5) Algoritma terbaik di setiap seed
terbaik = df.loc[df.groupby("seed")["val_f1"].idxmax(), ["seed", "algoritma", "val_f1"]]
print("\n=== Pemenang per seed (val_f1) ===")
print(terbaik.sort_values("seed").to_string(index=False))

=== val_f1 per seed ===
seed                         0      42     123    1337    3407    mean     std
algoritma                                                                     
fedavg                  0.4800  0.4737  0.4831  0.4731  0.4779  0.4776  0.0042
fedpairwise_T0.01_d0    0.4819  0.4813  0.4814  0.4767  0.4815  0.4805  0.0022
fedpairwise_T0.01_d0.3  0.4826  0.4823  0.4826  0.4746  0.4815  0.4807  0.0034
fedpairwise_T0.01_d0.5  0.4826  0.4804  0.4835  0.4766  0.4796  0.4805  0.0027
fedprox                 0.4808  0.4779  0.4844  0.4749  0.4786  0.4793  0.0035
fedsim_g3               0.4488  0.4636  0.4656  0.4542  0.4648  0.4594  0.0075
fedsim_g5               0.4650  0.4636  0.4729  0.4606  0.4695  0.4663  0.0049

=== Semua metrik per algoritma & seed ===
                             val_f1  val_auc  val_prauc  val_prec  val_rec
algoritma              seed                                               
fedavg                 0     0.4800   0.7408     0.4357    0.3950   0.61

In [3]:
# Pemetaan nama metrik -> kolom di CSV
metrik = {
    "Recall":  "val_rec",
    "F1":      "val_f1",
    "AUC-ROC": "val_auc",
    "PR-AUC":  "val_prauc",
}

# 1) Run tunggal terbaik untuk tiap metrik (algoritma + seed)
baris = []
for nama, kol in metrik.items():
    r = df.loc[df[kol].idxmax()]
    baris.append({"metrik": nama, "algoritma": r["algoritma"],
                  "seed": r["seed"], "nilai": r[kol]})
print("=== Kinerja tertinggi (run tunggal) ===")
print(pd.DataFrame(baris).round(4).to_string(index=False))

# 2) Nilai maksimum tiap algoritma lintas seed
maks = df.groupby("algoritma")[list(metrik.values())].max()
maks.columns = list(metrik.keys())
print("\n=== Nilai maksimum per algoritma ===")
print(maks.round(4))

# 3) Rata-rata lintas seed (lebih adil daripada satu run beruntung)
rata = df.groupby("algoritma")[list(metrik.values())].mean()
rata.columns = list(metrik.keys())
print("\n=== Rata-rata per algoritma lintas seed ===")
print(rata.round(4))

# 4) Algoritma terbaik berdasarkan rata-rata, per metrik
print("\n=== Algoritma terbaik berdasarkan rata-rata ===")
for nama in metrik:
    top = rata[nama].idxmax()
    print(f"{nama:8s}: {top}  ({rata.loc[top, nama]:.4f})")

# 5) Top-3 run untuk tiap metrik
for nama, kol in metrik.items():
    print(f"\n=== Top 3 run berdasarkan {nama} ===")
    print(df.nlargest(3, kol)[["algoritma", "seed", kol]].round(4).to_string(index=False))

# 6) Tabel ringkas: rata-rata ± std, diurutkan berdasarkan F1
ringkas = df.groupby("algoritma")[list(metrik.values())].agg(["mean", "std"])
ringkas.columns = [f"{nama}_{stat}" for nama, kol in metrik.items() for stat in ("mean", "std")]
print("\n=== Ringkasan mean & std ===")
print(ringkas.sort_values("F1_mean", ascending=False).round(4))

=== Kinerja tertinggi (run tunggal) ===
 metrik            algoritma  seed  nilai
 Recall              fedprox   123 0.6391
     F1              fedprox   123 0.4844
AUC-ROC fedpairwise_T0.01_d0  3407 0.7445
 PR-AUC fedpairwise_T0.01_d0  3407 0.4462

=== Nilai maksimum per algoritma ===
                        Recall      F1  AUC-ROC  PR-AUC
algoritma                                              
fedavg                  0.6325  0.4831   0.7425  0.4398
fedpairwise_T0.01_d0    0.6327  0.4819   0.7445  0.4462
fedpairwise_T0.01_d0.3  0.6325  0.4826   0.7444  0.4457
fedpairwise_T0.01_d0.5  0.6329  0.4835   0.7443  0.4449
fedprox                 0.6391  0.4844   0.7435  0.4415
fedsim_g3               0.6171  0.4656   0.7262  0.4147
fedsim_g5               0.6172  0.4729   0.7314  0.4221

=== Rata-rata per algoritma lintas seed ===
                        Recall      F1  AUC-ROC  PR-AUC
algoritma                                              
fedavg                  0.6206  0.4776   0.7395  0.